# PC-FNO3D national — règle GFS v5.2, partie C (Colab L4) : étapes 1 + 3 (validation 2021)

Règle `configs/national/regle_national_gfs_v5_2.json` (sha256 `542e676c90da…`), approuvée par Ines le 23/11/2026 et vérifiée par le carnet ; elle remplace la v5.1 (arrêtée sans verdict le 23/11 par son contrôle d'échéances, avant tout calcul aux aéroports) sur ce seul point. La partie A (GFS) et l'affinage C1 (époque 2) de la v5.1 sont réutilisés tels quels. Code : dépôt public au commit `45842e7e1647` (le carnet refuse de tourner tant qu'il n'est pas renseigné). Sorties : `MyDrive/pcfno3d_national/gfs_v5_regle/validation_2021_v5_2/` puis `resultat/` (dossiers neufs, rien n'est supprimé ni écrasé). Choix sur la validation 2021 seulement ; 2022 n'est pas relu ; l'échantillon 2026 (jours impairs, 00/06/12/18 UTC, ~608 instants) n'est lu qu'une fois, et seulement si une variante passe les trois cases sur 2021 puis si les parties vent de l'entrée de production sont identiques à celles du cache. Un instant sans l'une des trois échéances GFS est retiré pour toutes les sources et publié. GFS et ERA5 ne sont jamais appelés mesures (seuls les METAR le sont). Aucun bruit ajouté. Plafond : 90 unités Colab pour v5.1 + v5.2 ; le coût affiché est une estimation. La session est libérée à la fin.

## 0. Commit figé, budget, GPU (L4 ou A100, au moins 22 Go)

In [ ]:
# Commit FIGE du code (depot public). Tant qu'il n'est pas renseigne, rien ne tourne : session liberee.
import re as _re
COMMIT_FIGE = "45842e7e16477c2583ebc1684ed00490cff726b5"
if not _re.fullmatch(r"[0-9a-f]{40}", COMMIT_FIGE):
    from google.colab import runtime
    print("commit du code non renseigne (scripts/make_national_gfs_v5_nb.py, COMMIT) : rien n'est lance, "
          "session liberee")
    runtime.unassign()
    raise SystemExit("commit non fige")
print("commit fige :", COMMIT_FIGE)

In [ ]:
# BUDGET (regle v5.2) : plafond 90 unites Colab pour toute la famille v5.1 + v5.2 (A + B1 de la v5.1, ~17
# unites le 23/11, puis C + B2). Renseigner les unites DEJA consommees (lues sur le solde Colab). Cout : ESTIMATION.
UNITES_DEJA_UTILISEES = None
_mat, _taux, _hmin, _hmax = "L4", 4.8, 1.0, 2.5
_bas, _haut = _taux * _hmin, _taux * _hmax
print(f"cout ESTIME de la partie C (L4, ~4.8 unites/h, estimation) : {_hmin:g}-{_hmax:g} h, "
      f"soit ~{_bas:.0f}-{_haut:.0f} unites ; plafond : 90 unites (v5.1 + v5.2)")
if UNITES_DEJA_UTILISEES is None or float(UNITES_DEJA_UTILISEES) + _haut > 90:
    from google.colab import runtime
    print("budget : UNITES_DEJA_UTILISEES non renseigne, ou plafond risque d'etre depasse : rien n'est lance, "
          "demander a Ines (regle v5.2, « budget ») ; session liberee")
    runtime.unassign()
    raise SystemExit("budget non verifie")
print(f"budget : {float(UNITES_DEJA_UTILISEES):g} deja utilisees + {_haut:.0f} au plus <= 90")

In [ ]:
# Inference seulement (le national tient sur L4 en service). GPU d'au moins 22 Go (L4 ou A100) exige ;
# sinon la session est liberee AVANT le montage du Drive.
import subprocess as _sp
_g = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
             capture_output=True, text=True)
_nom, _mo = ([s.strip() for s in _g.stdout.strip().splitlines()[0].split(",")] if _g.returncode == 0
             and _g.stdout.strip() else ("aucun GPU", "0"))
print("GPU attribue :", _nom, f"{float(_mo) / 1024:.1f} Go")
if float(_mo) < 22 * 1024 or not any(k in _nom for k in ("L4", "A100")):
    from google.colab import runtime
    print("L4 ou A100 (>= 22 Go) exige : session liberee, rien n'a ete lance.")
    runtime.unassign()
    raise SystemExit("GPU L4 / A100 de 22 Go au moins exige")

## 1. Drive, code au commit figé, données ; refus si la sélection v5.2 existe déjà

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "45842e7e16477c2583ebc1684ed00490cff726b5"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"), ("regle_national_gfs_v4.json", "4ca1f2a0c34856d6840d620cd9457534c37cb316a59080400e2853e2f0d7d721"), ("regle_national_gfs_v5_1.json", "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd"), ("regle_national_gfs_v5_2.json", "542e676c90da2a6440ef3e64f323f8ebc3cf41f9926777934db54f5827583cb0")):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regles v4, gfs_v4, gfs_v5_1 et gfs_v5_2 verifiees")

GV5 = NAT / "gfs_v5_regle"                 # dossier de la famille v5 (rien d'autre n'y ecrit)
ECH = (0, 24, 48)
F = {e: GV5 / f"forcage_gfs_f{e}.csv" for e in ECH}                 # entrainement 2023-2025 + validation 2021
TN = {e: GV5 / f"gfs_f{e}_tn" for e in ECH}                         # echeances telechargees (boite nationale)
F26 = {e: GV5 / "jugement_2026" / f"forcage_gfs_2026_f{e}.csv" for e in ECH}
TN26 = {e: GV5 / f"gfs_2026_f{e}_tn" for e in ECH}
# archives deja telechargees, LUES SEULEMENT (rien n'y est ecrit) : OS7 du Grand Tunis, OS7 national s'il existe
LECTURE = {e: [d for d in (GT / "os7" / f"gfs_f{e}_instants", NAT / "os7" / f"gfs_f{e}_tn") if d.exists()] for e in ECH}
VAL = GV5 / "validation_2021_v5_2"         # etapes 1 + 3 de la v5.2 (dossier NEUF ; validation_2021/ = arret v5.1)
RES = GV5 / "resultat"                     # etape 4 (controle, puis jugement 2026)
METAR26 = GV5 / "metar_2026"               # METAR 2026 (archive IEM), telecharges seulement a l'etape 4
RCP = NAT / "runs" / "national_1km_v4_Cp_s44"          # C' s44 tel que livre (lu seulement)
R5 = NAT / "runs" / "national_1km_v5_gfs"              # C1 : entraine le 23/11 sous la v5.1, lu seulement
SHA_REGLE = "542e676c90da2a6440ef3e64f323f8ebc3cf41f9926777934db54f5827583cb0"                              # regle executee : v5.2
SHA_V5_1 = "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd"                                # marqueur de C1
print("accord :", "Ines, 23/11/2026 : regle_national_gfs_v5_2 approuvee et hachee (542e676c...)")

In [ ]:
def refuser(raison):
    from google.colab import runtime
    print("partie C refusee :", raison, "; session liberee")
    runtime.unassign()
    raise SystemExit(raison)

# Partie C : refus AVANT toute extraction si la selection v5.2 existe deja (une seule selection, rien n'est ecrase)
if (VAL / "selection.json").exists():
    refuser(f"{VAL} contient deja une selection")
for _e in ECH:
    if not F[_e].exists():
        refuser(f"{F[_e]} absent : la partie A (v5.1) doit etre faite")

# cache v3 : seulement _S et _E des instants de VALIDATION, jeu v3 (stations, grille)
CACHE = Path("/content/national/cache")
CACHE.mkdir(parents=True, exist_ok=True)
import pandas as pd
from pcfno3d.dataset import is_synoptic
parts = []
for k in (1, 2, 3):
    with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
        with zf.open(f"index_partie_{k}.csv") as fi:
            parts.append(pd.read_csv(fi))
idx = pd.concat(parts).drop_duplicates("stem")
idx.to_csv(CACHE / "index.csv", index=False)
val = idx[(idx["split"] == "val") & idx["timestamp"].map(is_synoptic)]
voulus = {f"{s}_{x}.npy" for s in val["stem"] for x in ("S", "E")}
for k in (1, 2, 3):
    with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
        for m in zf.namelist():
            if Path(m).name in voulus and not (CACHE / m).exists():
                zf.extract(m, CACHE)
print(len(val), "instants synoptiques de validation,", sum(1 for _ in CACHE.rglob("*_S.npy")), "fichiers _S extraits")
JEU = Path("/content/national/jeu")
if not (JEU / "meta.json").exists():
    JEU.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)

## 2. Vérification de C1, puis diagnostic et choix sur 2021 ; fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import yaml, shlex
import pandas as pd
def preparer_forcages_et_configs():
    """Forcages GFS (entrainement + validation) sur le disque local, configuration de C' s44 tel que livre."""
    for e in ECH:
        if not F[e].exists():
            raise FileNotFoundError(f"{F[e]} absent : la partie A (v5.1) doit etre faite")
    locf = G / "gfs_v5"                            # le chemin lu par configs/national_v5_gfs_mixte_colab.yaml
    locf.mkdir(exist_ok=True)
    for e in ECH:
        for suf in (".csv", ".npz"):
            if not (locf / F[e].with_suffix(suf).name).exists():
                shutil.copy2(F[e].with_suffix(suf), locf / F[e].with_suffix(suf).name)
    c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))     # C' s44 tel que livre
    c["name"] = "national_v4_eval"
    c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RCP), "/content/run_local_v4_sCp_s44"
    yaml.safe_dump(c, open(CODE / "configs" / "national_v4_eval.yaml", "w"), sort_keys=False)
    return locf
def verifier_c1():
    """C1 REUTILISE (regle v5.2, etape 2, deja_fait) : marqueur de la v5.1, 12 epoques finies, best.pt = epoque 2."""
    import torch
    m = R5 / "regle_gfs_v5_1.json"
    if not m.exists() or json.loads(m.read_text()).get("regle_sha256") != SHA_V5_1:
        raise SystemExit(f"{m} absent ou d'une autre regle : ce n'est pas le C1 de la v5.1")
    if not any(r.get("epoch") == 12 - 1 for r in json.loads((R5 / "history.json").read_text())):
        raise SystemExit("affinage C1 non termine (12 epoques) : rien n'est relance ici")
    ck = choisir_poids(R5)
    ep = torch.load(str(ck), map_location="cpu", mmap=True, weights_only=False).get("epoch", -2) + 1
    if ep != 2:
        raise SystemExit(f"{ck} est l'epoque {ep} ; la regle v5.2 dit 2 : verifier sur le Drive")
    print("C1 verifie :", ck, "| epoque", ep, "| marqueur de la v5.1")
    return ck

# PARTIE C (Colab L4) : etapes 1 + 3 sur la validation 2021 (13 variantes x 3 cases) ; rien de 2022 ni de 2026
from google.colab import runtime
import traceback
try:
    LOCF = preparer_forcages_et_configs()
    INIT = choisir_poids(RCP)
    INIT_C1 = verifier_c1()
    print("C' s44 :", INIT, "| C1 :", INIT_C1)
    lancer("choisir_gfs_v5_2", f"python -u scripts/choisir_national_gfs_v5.py --cp national_v4_eval --c1 national_v5_gfs_mixte_colab "
                               f"--gfs {LOCF} --obs {G / 'obs'} --sortie {VAL} "
                               f"--checkpoint-cp {shlex.quote(str(INIT))} --checkpoint-c1 {shlex.quote(str(INIT_C1))}")
    sel = json.loads((VAL / "selection.json").read_text())
    print((VAL / "validation_gfs_v5.md").read_text())
    if sel["arret"]:
        print("ARRET (regle, etape 3) : aucune variante ne passe les trois cases sur 2021 ; 2026 n'est pas lu, "
              "le carnet B2 refusera de tourner")
    else:
        print("variante retenue :", sel["retenue"], "-> carnet B2 (L4), apres l'accord d'Ines et le budget")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()